# Colab 32 — pooling x objective 2x2 ablation (SNNEED, no PLMs)

**Why:** the colab15->colab16 jump changed *two* things at once — the **objective**
(band-weighted MSE distance-regression -> 3-bin cross-entropy classifier) **and** the **encoder**
(flatten -> `AdaptiveAvgPool1d(K=16)`). That supports *"the combined revision worked"* but **cannot**
attribute the gain to the classifier or to pooling separately. This notebook runs the full **2x2** so the
two effects are isolated:

| | no-pool (flatten) | pool (AdaptiveAvgPool K=16) |
|---|---|---|
| **regression** (band-weighted MSE, distance readout) | reg·noPool  *(≈ colab15)* | reg·pool |
| **classifier** (3-bin CE on \|e_a−e_b\|) | clf·noPool | clf·pool  *(= SNNEED, colab16)* |

All four are trained on the **same 30k synthetic-AA pairs** (identical data within a seed; only the encoder
and the objective differ) and evaluated **identically**: discard any head, embed with the encoder, score
pairs by cosine, and report **Spearman / AUROC / MAP@10** on **synth / 3Di / SS / AA**. SNNEED-only — no
ESM-2 / ProtT5. Reads no result CSVs; writes receipts only.

**Reads out:** the two main effects (Δ from pooling, Δ from the objective) and their interaction, per
feed and metric — the evidence needed to say *"pooling contributed X, the objective contributed Y."*

## 1. Setup (self-contained — clones the repo for the CATH data)

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')

In [ ]:
DATA_DIR = '/content/thesis-edit-distance-nn/sampledata/cath'
for f in ['cath_s20_train70.csv.gz', 'cath_s20_test30.csv.gz', 'cath_s20_3di.csv.gz']:
    p = os.path.join(DATA_DIR, f); print(f'{"OK" if os.path.exists(p) else "MISSING":<8} {p}')

In [ ]:
!pip install torch rapidfuzz scikit-learn scipy matplotlib --quiet

In [ ]:
import time, itertools, numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from rapidfuzz.distance import Levenshtein as RFLev
from rapidfuzz.process import cdist as rf_cdist
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

# ---- config ----
N_TRAIN = 30_000                 # shared synthetic-AA training pairs per seed (matches colab16/deck)
SEEDS   = [0, 1, 2]              # set to [0] for a ~3x faster single-seed pass
EPOCHS  = 30
STRAT_PER_BIN = 400
STRAT_CAND    = 200_000
SYN_PERTURB, SYN_INDEP = 20_000, 8_000   # synthetic eval feed (disjoint from training)

FEED_ORDER = ['synth', '3Di', 'SS', 'AA']
FEED_COLOR = {'synth': '#ff7f0e', '3Di': '#1f77b4', 'SS': '#d62728', 'AA': '#7f7f7f'}
CONFIGS = [('reg', 'nopool'), ('reg', 'pool'), ('clf', 'nopool'), ('clf', 'pool')]
CONFIG_LABEL = {('reg', 'nopool'): 'reg·noPool', ('reg', 'pool'): 'reg·pool',
                ('clf', 'nopool'): 'clf·noPool', ('clf', 'pool'): 'clf·pool (SNNEED)'}
CONFIG_COLOR = {('reg', 'nopool'): '#9e9e9e', ('reg', 'pool'): '#7bb0df',
                ('clf', 'nopool'): '#f0a860', ('clf', 'pool'): '#c026a6'}

## 2. Constants, helpers, and the four architectures

In [ ]:
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'; SS_ALPHABET = 'HLS'
CHAR_TO_IDX = {c: i for i, c in enumerate(AA_ALPHABET)}; PAD_IDX = 20; VOCAB = 21
MIN_LEN, MAX_LEN, BS, K = 50, 200, 128, 16
BAND_LOW_AA, BAND_HIGH = 0.30, 0.70
AA_SET, SS_SET = set(AA_ALPHABET), set(SS_ALPHABET)
is_aa = lambda s: all(c in AA_SET for c in s); is_ss = lambda s: all(c in SS_SET for c in s)
def norm_lev(a, b):
    L = max(len(a), len(b)); return 1.0 if L == 0 else 1.0 - RFLev.distance(a, b) / L
def encode_pad(seq):
    idx = [CHAR_TO_IDX[c] for c in seq][:MAX_LEN]; idx += [PAD_IDX]*(MAX_LEN-len(idx))
    return torch.tensor(idx, dtype=torch.long)
def perturb(seq, k, abc, rng):
    s = list(seq); abc = list(abc)
    for _ in range(k):
        if len(s) == 0: op = 'ins'
        elif len(s) >= MAX_LEN: op = rng.choice(['sub', 'del'])
        else: op = rng.choice(['sub', 'ins', 'del'])
        if op == 'sub': i = rng.integers(0, len(s)); s[i] = rng.choice([c for c in abc if c != s[i]])
        elif op == 'ins': i = rng.integers(0, len(s)+1); s.insert(i, rng.choice(abc))
        else: i = rng.integers(0, len(s)); del s[i]
    return ''.join(s)
def rand_seq(abc, rng): L = int(rng.integers(MIN_LEN, MAX_LEN+1)); return ''.join(rng.choice(list(abc), size=L))
def bin_idx(x, band_low): return 0 if x < band_low else (1 if x < BAND_HIGH else 2)

# --- two encoders: identical except the length-collapse step ---
class EncPool(nn.Module):     # AdaptiveAvgPool(K)  (colab16 encoder)
    def __init__(s):
        super().__init__(); s.emb = nn.Embedding(VOCAB, 32, padding_idx=PAD_IDX)
        s.c1 = nn.Conv1d(32, 32, 3, padding=1); s.c2 = nn.Conv1d(32, 64, 3, padding=1)
        s.pool = nn.AdaptiveAvgPool1d(K); s.fc = nn.Linear(64*K, 128)
    def forward(s, x):
        m = (x != PAD_IDX).float(); e = s.emb(x).permute(0, 2, 1)
        h = F.relu(s.c1(e)); h = F.relu(s.c2(h)); h = h * m.unsqueeze(1)
        return F.normalize(s.fc(s.pool(h).flatten(1)), p=2, dim=1)
class EncNoPool(nn.Module):   # flatten full length  (colab15 encoder)
    def __init__(s):
        super().__init__(); s.emb = nn.Embedding(VOCAB, 32, padding_idx=PAD_IDX)
        s.c1 = nn.Conv1d(32, 32, 3, padding=1); s.c2 = nn.Conv1d(32, 64, 3, padding=1)
        s.fc = nn.Linear(64*MAX_LEN, 128)
    def forward(s, x):
        m = (x != PAD_IDX).float(); e = s.emb(x).permute(0, 2, 1)
        h = F.relu(s.c1(e)); h = F.relu(s.c2(h)); h = h * m.unsqueeze(1)
        return F.normalize(s.fc(h.flatten(1)), p=2, dim=1)
ENCODERS = {'pool': EncPool, 'nopool': EncNoPool}

# --- two objectives wrapping a given encoder ---
class ClfModel(nn.Module):    # 3-bin CE on |e_a - e_b|; head discarded at inference
    def __init__(s, enc):
        super().__init__(); s.encoder = enc
        s.head = nn.Sequential(nn.Linear(128, 64), nn.LeakyReLU(0.01), nn.Linear(64, 3))
    def forward(s, a, b): return s.head(torch.abs(s.encoder(a) - s.encoder(b)))
class RegModel(nn.Module):    # band-weighted MSE on the distance readout 1 - ||d||/2
    def __init__(s, enc):
        super().__init__(); s.encoder = enc
    def forward(s, a, b):
        ea, eb = s.encoder(a), s.encoder(b)
        return 1.0 - torch.linalg.vector_norm(ea - eb, ord=2, dim=1) / 2.0

class DS(Dataset):
    def __init__(s, pp, obj): s.p = pp; s.obj = obj
    def __len__(s): return len(s.p)
    def __getitem__(s, i):
        a, b, l = s.p[i]
        y = torch.tensor(bin_idx(l, BAND_LOW_AA)) if s.obj == 'clf' else torch.tensor(l, dtype=torch.float32)
        return encode_pad(a), encode_pad(b), y

def band_w(y):                # colab15 band weighting: down-weight far, up-weight high
    w = torch.full_like(y, 2.0); w[y < BAND_LOW_AA] = 0.5; w[y >= BAND_HIGH] = 4.0; return w

def build_pairs(n, seed):
    rng = np.random.default_rng(seed); pairs = []
    while len(pairs) < n:
        sd = rand_seq(AA_ALPHABET, rng); L = len(sd); t = float(rng.uniform(0, 1)); k = max(0, int(round((1-t)*L)))
        o = perturb(sd, k, AA_ALPHABET, rng)
        if 1 <= len(o) <= MAX_LEN: pairs.append((sd, o, norm_lev(sd, o)))
    return pairs

def train_config(obj, enc_name, pairs, seed, label):
    torch.manual_seed(seed)
    model = (ClfModel(ENCODERS[enc_name]()) if obj == 'clf' else RegModel(ENCODERS[enc_name]())).to(device)
    dl = DataLoader(DS(pairs, obj), batch_size=BS, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), 1e-3); crit = nn.CrossEntropyLoss()
    model.train()
    for ep in range(1, EPOCHS+1):
        tot = nb = 0
        for a, b, y in dl:
            a, b, y = a.to(device), b.to(device), y.to(device)
            if obj == 'clf':
                loss = crit(model(a, b), y)
            else:
                pred = model(a, b); loss = (band_w(y) * (pred - y)**2).mean()
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item(); nb += 1
        if ep % 10 == 0 or ep == 1: print(f'    [{label}] epoch {ep}/{EPOCHS} loss {tot/nb:.4f}')
    if device.type == 'cuda': torch.cuda.synchronize()
    model.eval(); return model

print('encoder params  pool:', sum(p.numel() for p in EncPool().parameters()),
      ' nopool:', sum(p.numel() for p in EncNoPool().parameters()))

## 3. Real pools (AA/SS/3Di) + oracles + stratified pairs + synth feed (built once)

In [ ]:
raw = pd.concat([pd.read_csv(f'{DATA_DIR}/cath_s20_train70.csv.gz'),
                 pd.read_csv(f'{DATA_DIR}/cath_s20_test30.csv.gz')],
                ignore_index=True).drop_duplicates('domain_id')
seqs3 = pd.read_csv(f'{DATA_DIR}/cath_s20_3di.csv.gz')
RESCUED = {'4z0mC02', '3qkaE02'}
def _valid(seq, isstd, d):
    return (isinstance(seq, str) and isstd(seq) and ((MIN_LEN <= len(seq) <= MAX_LEN) or d in RESCUED))
id_to_aa  = {d: s for d, s in zip(raw['domain_id'], raw['aa_seq'])              if _valid(s, is_aa, d)}
id_to_ss  = {d: s for d, s in zip(raw['domain_id'], raw['ss_seq'])              if _valid(s, is_ss, d)}
id_to_3di = {d: s for d, s in zip(seqs3['domain_id'], seqs3['3di'].astype(str)) if _valid(s, is_aa, d)}
LOOK = {'AA': id_to_aa, 'SS': id_to_ss, '3Di': id_to_3di}
POOL_SEQ = {f: list(LOOK[f].values()) for f in LOOK}
CATH_FEEDS = ['AA', 'SS', '3Di']
for f in CATH_FEEDS: print(f'  {f:<4} pool = {len(POOL_SEQ[f]):>6}')

In [ ]:
def build_oracle(feed, block=1024):
    seqs = POOL_SEQ[feed]; lens = np.array([len(s) for s in seqs]); N = len(seqs)
    T_high = {}; pos_pairs = []
    for r0 in range(0, N, block):
        r1 = min(r0 + block, N)
        Dm = rf_cdist(seqs[r0:r1], seqs, scorer=RFLev.distance, workers=-1).astype(np.float64)
        den = np.maximum(lens[r0:r1][:, None], lens[None, :]); den[den == 0] = 1
        sim = 1.0 - Dm / den
        for a in range(r1 - r0):
            i = r0 + a; row = sim[a].copy(); row[i] = -1.0
            hi = np.where(row >= BAND_HIGH)[0]
            if hi.size: T_high[i] = hi.astype(np.int32)
            for j in hi:
                if j > i: pos_pairs.append((i, int(j), float(row[j])))
    return dict(T_high=T_high, pos_pairs=pos_pairs)
ORACLE = {}
for f in CATH_FEEDS:
    print(f'Building {f} oracle (SS is the slow one)...'); ORACLE[f] = build_oracle(f)
    print(f'  {f}: queries@0.70={len(ORACLE[f]["T_high"]):>6}, pos pairs={len(ORACLE[f]["pos_pairs"]):>7}')

In [ ]:
def build_strat_pairs(feed, rng):
    seqs = POOL_SEQ[feed]; N = len(seqs)
    a = rng.integers(0, N, STRAT_CAND); b = rng.integers(0, N, STRAT_CAND)
    keep = a != b; a, b = a[keep], b[keep]
    nl = np.array([norm_lev(seqs[i], seqs[j]) for i, j in zip(a, b)])
    if feed in ORACLE and ORACLE[feed]['pos_pairs']:
        parr = np.array(ORACLE[feed]['pos_pairs'], dtype=float)
        a = np.concatenate([a, parr[:, 0].astype(np.int64)]); b = np.concatenate([b, parr[:, 1].astype(np.int64)])
        nl = np.concatenate([nl, parr[:, 2]])
    bins = np.clip(np.digitize(nl, np.linspace(0, 1, 11)) - 1, 0, 9)
    ai, aj, av = [], [], []
    for bb in range(10):
        idx = np.where(bins == bb)[0]
        if idx.size == 0: continue
        take = rng.permutation(idx)[:STRAT_PER_BIN]; ai.append(a[take]); aj.append(b[take]); av.append(nl[take])
    return dict(i=np.concatenate(ai).astype(np.int64), j=np.concatenate(aj).astype(np.int64), nl=np.concatenate(av))
STRAT = {f: build_strat_pairs(f, np.random.default_rng(999)) for f in CATH_FEEDS}

def build_synth_feed(n_perturb, n_indep, per_bin=STRAT_PER_BIN, seed=20260810):
    r = np.random.default_rng(seed); recs = []
    for _ in range(n_perturb):
        base = rand_seq(AA_ALPHABET, r); part = perturb(base, int(r.integers(0, len(base)+1)), AA_ALPHABET, r)
        if 1 <= len(part) <= MAX_LEN: recs.append((base, part))
    for _ in range(n_indep):
        recs.append((rand_seq(AA_ALPHABET, r), rand_seq(AA_ALPHABET, r)))
    recs = [(a, b, norm_lev(a, b)) for a, b in recs]
    nl_all = np.array([x[2] for x in recs]); bins = np.clip(np.digitize(nl_all, np.linspace(0, 1, 11)) - 1, 0, 9)
    take = []
    for bb in range(10):
        idx = np.where(bins == bb)[0]
        if idx.size: take.extend(r.permutation(idx)[:per_bin].tolist())
    seqs, I, J, NL = [], [], [], []
    for idx in take:
        a, b, nl = recs[int(idx)]; I.append(len(seqs)); seqs.append(a); J.append(len(seqs)); seqs.append(b); NL.append(nl)
    return seqs, np.array(I), np.array(J), np.array(NL)
SYN_SEQ, SYN_I, SYN_J, SYN_NL = build_synth_feed(SYN_PERTURB, SYN_INDEP)
POOL_SEQ['synth'] = SYN_SEQ
ORACLE['synth'] = build_oracle('synth')
print(f'synth: pool={len(SYN_SEQ)}, pairs={len(SYN_NL)}, queries@0.70={len(ORACLE["synth"]["T_high"])}')

## 4. Metric helpers (identical eval for all four configs: encoder -> cosine)

In [ ]:
@torch.no_grad()
def embed_torch(model, seqs, bs=256):
    outs = []
    for i in range(0, len(seqs), bs):
        x = torch.stack([encode_pad(s) for s in seqs[i:i+bs]]).to(device)
        outs.append(model.encoder(x))
    return torch.cat(outs)
def _auroc(sim, nl):
    y = (nl >= BAND_HIGH).astype(int)
    return roc_auc_score(y, sim) if 0 < y.sum() < len(y) else np.nan
def map10_torch(E, T_high, k=10, qb=256):
    q = list(T_high.keys())
    if not q: return np.nan
    aps = []
    for s0 in range(0, len(q), qb):
        qi = q[s0:s0+qb]; sc = E[qi] @ E.t()
        for r, idx in enumerate(qi): sc[r, idx] = -1e9
        top = torch.topk(sc, k, dim=1).indices.cpu().numpy()
        for r, idx in enumerate(qi):
            ts = set(T_high[idx].tolist()); hits = 0; ap = 0.0
            for rr, o in enumerate(top[r], 1):
                if o in ts: hits += 1; ap += hits / rr
            aps.append(ap / min(len(ts), k))
    return float(np.mean(aps))
def eval_feed(model, feed):
    E = embed_torch(model, POOL_SEQ[feed]); Enp = E.cpu().numpy()
    if feed == 'synth':
        sim = np.sum(Enp[SYN_I] * Enp[SYN_J], axis=1); nl = SYN_NL
    else:
        P = STRAT[feed]; sim = np.sum(Enp[P['i']] * Enp[P['j']], axis=1); nl = P['nl']
    return spearmanr(sim, nl).correlation, _auroc(sim, nl), map10_torch(E, ORACLE[feed]['T_high'])

## 5. Run the 2x2 (four configs x seeds; identical data within a seed)

In [ ]:
rows = []
wall0 = time.perf_counter()
for seed in SEEDS:
    pairs = build_pairs(N_TRAIN, seed)          # shared across the four configs this seed
    print(f'seed {seed}: built {len(pairs)} shared training pairs')
    for obj, enc_name in CONFIGS:
        model = train_config(obj, enc_name, pairs, seed, CONFIG_LABEL[(obj, enc_name)])
        for feed in FEED_ORDER:
            sp, au, mp = eval_feed(model, feed)
            rows.append(dict(seed=seed, obj=obj, enc=enc_name, feed=feed,
                             spearman=sp, auroc=au, map10=mp))
        r = rows[-len(FEED_ORDER):]
        print('    ' + CONFIG_LABEL[(obj, enc_name)] + '  ' +
              '  '.join(f'{d["feed"]}:MAP={d["map10"]:.2f}' for d in r))
res = pd.DataFrame(rows)
res.to_csv('colab32_2x2_metrics.csv', index=False)
print(f'\nTotal wall-clock: {time.perf_counter()-wall0:.1f}s')

## 6. The 2x2 tables + conditional-effect decomposition

In [ ]:
METRICS = [('spearman', 'Spearman'), ('auroc', 'AUROC(>=0.70)'), ('map10', 'MAP@10')]
def grid(feed, metric):
    return (res[res.feed == feed].groupby(['obj', 'enc'])[metric].mean()
            .unstack('enc').reindex(index=['reg', 'clf'], columns=['nopool', 'pool']))
for metric, name in METRICS:
    print('=' * 60); print(f'{name} — 2x2 (rows=objective, cols=pooling), mean over seeds'); print('=' * 60)
    for feed in FEED_ORDER:
        g = grid(feed, metric)
        print(f'\n[{feed}]'); print(g.round(3).to_string())

# conditional (simple) effects per (feed, metric) — the honest, interpretable quantities.
# total lift SNNEED-vs-baseline decomposes two equivalent ways:
#   total = pool_in_reg + obj_in_pool = obj_in_noPool + pool_in_clf
eff = []
for metric, name in METRICS:
    for feed in FEED_ORDER:
        g = grid(feed, metric)
        rn, rp = g.loc['reg', 'nopool'], g.loc['reg', 'pool']
        cn, cp = g.loc['clf', 'nopool'], g.loc['clf', 'pool']
        eff.append(dict(metric=name, feed=feed, reg_noPool=rn, clf_pool=cp, total=cp - rn,
                        pool_in_reg=rp - rn, pool_in_clf=cp - cn,
                        obj_in_noPool=cn - rn, obj_in_pool=cp - rp,
                        interaction=(cp - cn) - (rp - rn)))
eff = pd.DataFrame(eff).round(3)
print('\n' + '=' * 96)
print('CONDITIONAL EFFECTS   total = pool_in_reg + obj_in_pool = obj_in_noPool + pool_in_clf ;  '
      'interaction = pool_in_clf − pool_in_reg')
print('=' * 96); print(eff.to_string(index=False))
eff.to_csv('colab32_2x2_effects.csv', index=False)

## 7. Figure — the four configs per metric and feed

In [ ]:
import matplotlib.pyplot as plt
def despine(ax): ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
xpos = np.arange(len(FEED_ORDER)); w = 0.2
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
for ax, (metric, name) in zip(axes, METRICS):
    for c, (obj, enc_name) in enumerate(CONFIGS):
        m = [res[(res.feed == f) & (res.obj == obj) & (res.enc == enc_name)][metric].mean() for f in FEED_ORDER]
        sd = [res[(res.feed == f) & (res.obj == obj) & (res.enc == enc_name)][metric].std() for f in FEED_ORDER]
        ax.bar(xpos + (c - 1.5) * w, m, w, yerr=sd, capsize=2,
               color=CONFIG_COLOR[(obj, enc_name)], label=CONFIG_LABEL[(obj, enc_name)])
    ax.set_xticks(xpos); ax.set_xticklabels(FEED_ORDER); ax.set_title(name); ax.set_ylim(bottom=min(0, ax.get_ylim()[0]))
    despine(ax)
axes[0].legend(fontsize=8, loc='lower left')
plt.tight_layout(); plt.savefig('colab32_2x2.png', dpi=150, bbox_inches='tight'); plt.show()

## 8. How to read this

Each cell of the 2x2 is one trained encoder; all four share the same 30k synthetic-AA data within a seed and
are scored the same way (encoder -> cosine). `total` is the full SNNEED-vs-baseline lift
(`clf·pool − reg·noPool`); it splits **two equivalent ways** along the edges of the square:

`total = pool_in_reg + obj_in_pool = obj_in_noPool + pool_in_clf`

- **pool_in_reg / pool_in_clf** — the effect of adding `AdaptiveAvgPool(K=16)` *within* each objective.
- **obj_in_noPool / obj_in_pool** — the effect of switching MSE->CE *within* each pooling choice.
- **interaction = pool_in_clf − pool_in_reg** — how much pooling's benefit *depends on* the objective (equal
  to how much the objective's benefit depends on pooling). If it is small, the two levers are separable and
  you may quote a single number for each; if it is large, report the effects **conditionally** and keep the
  claim as *"the combined revision worked"*.

Use this to replace the confounded *"the classifier and pooling improved the architecture"* with an
attributed statement per feed/metric. Real-AA stays the noise-level control (few natural high-sim pairs).
No PLMs here; this is the SNNEED-internal design ablation.

Outputs (written, not read): `colab32_2x2_metrics.csv`, `colab32_2x2_effects.csv`, `colab32_2x2.png`.